# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR<sup>2</sup> mass spectrometry dataset using the `mlcroissant` library.

**Dataset summary:** This dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples. It includes various fields such as accession, description, coverage percentage, peptide counts, and molecular weight (MW) alongside calculated parameters such as pI and normalized abundances across different samples.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{getattr(metadata, 'name', '')}: {getattr(metadata, 'description', '')}")
print(f"Published: {getattr(metadata, 'datePublished', '')}")
print(f"Identifier: {getattr(metadata, 'identifier', '')}")
print(f"Available record sets: {getattr(metadata, 'recordSet', []) if hasattr(metadata, 'recordSet') else 'N/A'}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We inspect the Croissant schema metadata to identify `RecordSet` objects and examine their fields and columns. All references to dataset entities use their `@id`.

---

In [ ]:
# Display the record sets with their @id and available fields
record_sets = getattr(metadata, 'recordSet', []) if hasattr(metadata, 'recordSet') else []
if not record_sets:
    # If record sets are not directly present, try to recover them from the metadata dict
    # This often happens if top-level keys are not class objects
    mdict = json.loads(metadata.to_json()) if hasattr(metadata, 'to_json') else metadata
    record_sets = mdict.get('recordSet', [])

record_set_ids = []
field_ids = {}

for rs in record_sets:
    # Each record set should be a dict with '@id' and 'field'
    rs_id = rs.get('@id', None)
    record_set_ids.append(rs_id)
    fields = rs.get('field', []) if 'field' in rs else []
    if fields:
        field_ids[rs_id] = [f['@id'] if isinstance(f, dict) else f for f in fields]
    print(f"RecordSet @id: {rs_id}")
    if fields:
        print(f" - Fields: {[f['@id'] if isinstance(f, dict) else f for f in fields]}")
    else:
        print(" - No field definitions found.")

if not record_set_ids:
    print("No record sets found in the schema.")

## 3. Data Extraction
Load data from specific record set(s) into DataFrames for analysis.

Use only the `@id` fields from the previous overview for referencing record sets and fields. For large datasets, preview only the first few records.

In [ ]:
# Extract records from all discovered record sets into Pandas DataFrames
dataframes = {}

# If no record sets are present, provide instructions.
if not record_set_ids:
    print("No record sets identified. Please check the Croissant schema for available data.")
else:
    for record_set_id in record_set_ids:
        print(f"Loading records for record set: {record_set_id}")
        try:
            records_gen = dataset.records(record_set=record_set_id)
            records = list(records_gen)
            if len(records) == 0:
                print(f"No data found for {record_set_id}.")
                continue
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Columns in {record_set_id}: {df.columns.tolist()}")
            display(df.head())
        except Exception as e:
            print(f"Failed to load data for {record_set_id}: {str(e)}")

# As demonstration, select the first detected record set (if available)
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Using main record set @id: {main_record_set_id}")
    if main_record_set_id in dataframes:
        df = dataframes[main_record_set_id]
        print(f"Fields: {df.columns.tolist()}")
        display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, and grouping on the selected record set.

- Select a numeric field for filtering and normalization (e.g. peptide count, MW, coverage_percent, etc.)
- Reference all columns/fields using their `@id` (as found previously).

In [ ]:
# --- EDA Section ---
import numpy as np

# Reference the DataFrame for the main record set
if not record_set_ids or main_record_set_id not in dataframes:
    print('No record set available for EDA.')
else:
    df = dataframes[main_record_set_id]
    print(f"Analyzing record set: {main_record_set_id}")

    # Select a numeric field by its @id. Replace with the actual field ID found in step 2/3.
    # We'll provide a fallback to look for common names.
    possible_numeric_fields = [
        'cr:field:coverage_percent',
        'cr:field:peptide_count',
        'cr:field:MW',
        'cr:field:pI',
        'cr:field:abundance',
        'cr:field:normalized_abundance'
    ]

    numeric_field = None
    for fid in possible_numeric_fields:
        if fid in df.columns:
            numeric_field = fid
            break

    if not numeric_field:
        # Fall back to auto-detect first numeric column
        for col in df.columns:
            if np.issubdtype(df[col].dtype, np.number):
                numeric_field = col
                break
    if not numeric_field:
        print("No numeric field for EDA found.")
    else:
        print(f"Selected numeric field for analysis: {numeric_field}")
        # Drop missing/NaN entries if present
        base_df = df[[numeric_field]].dropna()
        threshold = base_df[numeric_field].mean() if base_df[numeric_field].mean() < base_df[numeric_field].max() else base_df[numeric_field].median()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > {threshold:.2f} (using mean/median):")
        display(filtered_df.head())

        # Normalize values (z-score normalization)
        mean_ = base_df[numeric_field].mean()
        std_ = base_df[numeric_field].std()
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean_) / std_
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a categorical field. Look for likely group fields (e.g. sample type)
        possible_group_fields = [
            'cr:field:sample_type',
            'cr:field:accession',
            'cr:field:protein_class',
            'cr:field:group'
        ]
        group_field = None
        for f in possible_group_fields:
            if f in df.columns:
                group_field = f
                break
        if group_field and group_field in filtered_df.columns:
            # Compute mean of normalized value within groups
            grouped_df = filtered_df.groupby(group_field)[f"{numeric_field}_normalized"].mean()
            print(f"Grouped normalized {numeric_field} by {group_field}:\n", grouped_df.head())
        else:
            print("No suitable categorical group field found for grouping.")

## 5. Visualization
Visualize distributions and relationships in the data using Pandas and matplotlib/Seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# If EDA variables are available, visualize distribution
if 'filtered_df' in locals() and numeric_field in filtered_df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[numeric_field], kde=True, bins=30, color='steelblue')
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If normalized column present
    if f"{numeric_field}_normalized" in filtered_df.columns:
        plt.figure(figsize=(7,4))
        sns.histplot(filtered_df[f"{numeric_field}_normalized"], kde=True, bins=30, color='orange')
        plt.title(f'Normalized Distribution of {numeric_field}')
        plt.xlabel(f"{numeric_field}_normalized")
        plt.ylabel('Count')
        plt.show()

    # If group field found, also plot by group
    if 'group_field' in locals() and group_field and group_field in filtered_df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=30)
        plt.show()
else:
    print('No filtered DataFrame or numeric field available for visualization.')

## 6. Conclusion
This notebook demonstrates a reproducible workflow for loading, exploring, and processing the FAIR<sup>2</sup> mass spectrometry protein dataset using `mlcroissant`. By referencing all entities by their Croissant `@id`, the workflow is robust to schema changes and supports clear, FAIR-compliant analysis. Review your findings and adapt the analysis steps for your downstream ML or bioinformatics application.